# Task 2b: Specialist Shared Architecture Search (Optuna)

Conducts a shared architecture search on representative corruption data (Plan 6 §2b.2).
The locked winning architecture is then trained across all three specialists.

In [ ]:
# 1. Config
N_TRIALS = 20
OPTUNA_EPOCHS = 15
FULL_EPOCHS = 50
STUDY_NAME = "task2b_shared_architecture"
STORAGE_DB = "sqlite:///task2b_study.db"
SEED = 42

In [ ]:
# 2. Imports & Setup
import sys
from pathlib import Path
import yaml
import torch
from torch.utils.data import DataLoader
import optuna

RESEARCH_ROOT = Path("..").resolve()
if str(RESEARCH_ROOT) not in sys.path:
    sys.path.insert(0, str(RESEARCH_ROOT))

from constants import CONFIGS_ROOT
from src.device_utils import get_device, device_report
from src.datasets import PetDataset
from src.models_vae import ConvVAE
from src.train_loop import train_one_epoch_vae, validate_vae, set_seed
from src.checkpoint_utils import create_checkpoint_dict, save_checkpoint

set_seed(SEED)
device = get_device()

In [ ]:
# 3. Shared Architecture Objective
# Conducted on Salt & Pepper representative subset (Plan 6 §2b.2)
train_dataset = PetDataset(mode="train", corruption_mode="salt", seed=SEED)
val_dataset = PetDataset(mode="val", corruption_mode="salt", seed=SEED)

def objective(trial: optuna.Trial) -> float:
    lr = trial.suggest_float("lr", 1e-4, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    latent_dim = trial.suggest_categorical("latent_dim", [64, 128, 256])
    base_channels = trial.suggest_categorical("base_channels", [32, 64])
    dropout = trial.suggest_float("dropout", 0.0, 0.3)
    alpha = trial.suggest_float("alpha", 0.5, 0.95)
    beta = trial.suggest_float("beta", 1e-4, 1e-1, log=True)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    model = ConvVAE(
        in_channels=3,
        base_channels=base_channels,
        latent_dim=latent_dim,
        dropout=dropout,
        use_skip=False
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_loss = float("inf")

    for epoch in range(1, OPTUNA_EPOCHS + 1):
        train_one_epoch_vae(model, train_loader, optimizer, device, alpha, beta)
        vl = validate_vae(model, val_loader, device, alpha, beta)
        if vl["val_loss"] < best_loss:
            best_loss = vl["val_loss"]
        trial.report(vl["val_loss"], step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_loss

In [ ]:
# 4. Run Study & Export Configs
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_DB,
    load_if_exists=True,
    direction="minimize",
    pruner=optuna.pruners.MedianPruner()
)
study.optimize(objective, n_trials=N_TRIALS)

print(f"Winning Shared Architecture Params: {study.best_params}")

for ctype in ["salt", "blur", "occlusion"]:
    cfg_path = CONFIGS_ROOT / f"task2b_{ctype}_best_config.yaml"
    with open(cfg_path, "w") as f:
        yaml.dump({
            "task": f"task2b_specialist_{ctype}",
            "params": study.best_params,
            "best_val_loss": float(study.best_value)
        }, f, indent=2)
print("Saved configs for all 3 specialists.")